In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
import linear # type: ignore
import ploss # type: ignore
from common import test_checkgrad_2, test_compare_2 # type: ignore

In [ ]:
class Per_Lin_PLoss_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
        
    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()
        self.linear =  linear.Linear(in_features, out_features, W, b)

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        return z

    def loss(self, z, y):
        assert ((y == 1) | (y == -1)).all()
        
        L = tr.max(tr.zeros_like(z), -z * y).mean()
        return L

    def predict(self, x):
        with tr.no_grad():
            z = self.model(x)
            return z

In [ ]:
class Per_Lin_PLoss_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch’s autograd still orchestrates the overall gradient flow through the computational graph.
    """
    
    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()
        self.linear =  linear.Linear(in_features, out_features, W, b)

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        return z

    def loss(self, z, y):
        L = ploss.PLoss(reduction='mean')(z, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            z = self.model(x)
            return z

$$ m:\mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad z=m(w, b) = xw + b $$

$$ 
m: \mathbb{R}^m \times \mathbb{R}^n \to \mathbb{R}^n, \quad \mathbf{z}=
m(\mathbf{w}, \mathbf{b}) = X\mathbf{w} + \mathbf{b}
$$

$$ \frac{\partial \mathbf{z}}{\partial \mathbf{w}} = X $$

$$ \frac{\partial \mathbf{z}}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ 
d\mathbf{z} = 
\frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} 
$$

$$ d\mathbf{z} = X d\mathbf{w} + \mathbf{I}_{n} \, d\mathbf{b} $$

$$ 
f: \mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad L(z, t) = 
\begin{cases} 0 & \text{if } zt > 0 \\ -zt & \text{if } zt \leq 0 \end{cases} 
$$

$$ 
f:\mathbb{R^n \times \mathbb{R^n}} \to \mathbb{R^n}, \quad \mathbf{y} = 
L(\mathbf{z}, \mathbf{t}) = (f(z_1, t_1), f(z_2, t_2), \dots, f(z_n, t_n)) 
$$

$$ 
\frac{\partial L}{\partial z_i} = 
\begin{cases} 
    0 & \text{if } z_i t_i > 0 \\ 
    -t_i & \text{if } z_i t_i \leq 0 
\end{cases} 
$$

$$ \text{Frobenius equality} $$

$$ \left \langle \frac{dF}{d \mathbf{w}}, d\mathbf{w} \right \rangle + \left \langle \frac{dF}{d \mathbf{b}}, d\mathbf{b} \right \rangle = \left \langle \frac{dF}{dL}, dL \right \rangle $$

$$ \\[2em] $$

$$ \left \langle \frac{dF}{dL}, dL \right \rangle = \frac{dF}{dL} \left \langle \frac{dL}{d\mathbf{z}}, d\mathbf{z} \right \rangle =  $$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{d\mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{d\mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{d\mathbf{z}},\frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{d\mathbf{z}}, d\mathbf{w} \right \rangle + 
\frac{dF}{dL} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{d\mathbf{z}}, d\mathbf{b} \right \rangle = 
$$

$$ 
\left \langle \frac{dF}{dL} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{d\mathbf{z}}, d\mathbf{w} \right \rangle + 
\left \langle \frac{dF}{dL} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{d\mathbf{z}}, d\mathbf{b} \right \rangle \implies 
$$

$$ 
\begin{dcases}
\frac{dF}{d\mathbf{w}} = \frac{dF}{dL} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{d\mathbf{z}} = 1 \cdot X^\top \cdot \frac{dL}{d\mathbf{z}} \\
\\
\frac{dF}{d\mathbf{b}} = \frac{dF}{dL} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{d\mathbf{z}} = 1 \cdot I_n \cdot \frac{dL}{d\mathbf{z}}
\end{dcases}
$$


In [ ]:
class Per_Lin_PLoss_Gradient_Function(autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, t):
        z = x @ W + b
        
        mask = (z * t <= 0)
        L = tr.zeros_like(z)
        L[mask] = -z[mask] * t[mask]
        L = L.mean()

        ctx.save_for_backward(x, W, t, z)
        return L

    @staticmethod
    def backward(ctx, dF_dL):
        x, W, t, z = ctx.saved_tensors
        S, FI = x.shape
        FO = W.shape[1]
        N = S * FO

        dz_dw = x

        mask = (z * t <= 0)
        dL_dz = tr.zeros_like(z)
        dL_dz[mask] = -t[mask]        
        dL_dz = 1 / N * dL_dz

        dF_dW = dF_dL * dz_dw.t() @ dL_dz
        dF_db = dF_dL * dL_dz.sum(dim=0)

        return None, dF_dW, dF_db, None


class Per_Lin_PLoss_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """
    
    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `y`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None):
        super().__init__()

        # `W` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if W is None:
            self.weight = nn.Parameter(tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_PLoss_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            z = Per_Lin_PLoss_Gradient_Function._model(x, self.weight.t(), self.bias)
            return z
    
    def loss(self, p, y):
        with tr.no_grad():
            L = Per_Lin_PLoss_Gradient_Function._loss(p, y)
            return L

    def predict(self, x):
        with tr.no_grad():
            y = Per_Lin_PLoss_Gradient_Function.predict(x, self.weight.t(), self.bias)
            return y
        
    def forward(self, x, y):
        y = Per_Lin_PLoss_Gradient_Function.apply(x, self.weight.t(), self.bias, y)
        return y

In [ ]:
if __name__ == "__main__":
    scale = lambda y: (y >= 0.5).double() * 2 - 1

    test_checkgrad_2(Per_Lin_PLoss_Gradient_Function, 1, 1, 1, prec=0.01, fY=scale)
    test_checkgrad_2(Per_Lin_PLoss_Gradient_Function, 2, 2, 2, prec=0.01, fY=scale)
    test_checkgrad_2(Per_Lin_PLoss_Gradient_Function, 3, 3, 3, prec=0.01, fY=scale)

    scale = lambda y: (y >= 0.5).float() * 2 - 1

    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Backward, 1, 1, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Backward, 10, 1, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Backward, 10, 20, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Backward, 10, 20, 30, fY=scale)

    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Gradient, 1, 1, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Gradient, 10, 1, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Gradient, 10, 20, 1, fY=scale)
    test_compare_2(Per_Lin_PLoss_Autograd, Per_Lin_PLoss_Gradient, 10, 20, 30, fY=scale)
